In [1]:
import os

# path setting
current = os.getcwd()
while os.path.basename(current) != "Data_center_and_fossil_energy_Replication":
    current = os.path.dirname(current)

BASE_PATH = current

RAW = os.path.join(BASE_PATH, 'Data', 'raw')
TEMP = os.path.join(BASE_PATH, 'Data', 'temp')
USE = os.path.join(BASE_PATH, 'Data', 'use')
FIGURES = os.path.join(BASE_PATH, 'Results', 'Figures')
TABLES = os.path.join(BASE_PATH, 'Results', 'Tables')

for path in [RAW, TEMP, USE, FIGURES, TABLES]:
    os.makedirs(path, exist_ok=True)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Nature style settings
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 5,
    'axes.linewidth': 0.3,
})

# Read data
file_path = os.path.join(RAW, 'SPGlobal_Export.xlsx')
df_centers = pd.read_excel(file_path, sheet_name='Sheet1')
df_country = pd.read_excel(file_path, sheet_name='country')

# Filter valid samples: non-missing YR_BUILT, LATITUDE, LONGITUDE, and YR_BUILT != 2025
df_centers = df_centers[
    (df_centers['YR_BUILT'].notna()) & 
    (df_centers['LATITUDE'].notna()) & 
    (df_centers['LONGITUDE'].notna()) & 
    (df_centers['YR_BUILT'] != 2025)
].copy()

# Print number of valid samples
print(f"Number of valid samples: {len(df_centers)}")

# Merge data
df_merged = df_centers.merge(
    df_country[['COUNTRY', 'Alpha-3 code', 'Region']], 
    on='COUNTRY', 
    how='left'
)

# Filter valid year data (no year restriction)
df_time = df_merged[df_merged['YR_BUILT'].notna()].copy()

# Count data centers by country and year
country_year = df_time.groupby(['COUNTRY', 'YR_BUILT']).size().reset_index(name='count')

# Create pivot table
pivot = country_year.pivot_table(
    index='COUNTRY',
    columns='YR_BUILT',
    values='count',
    fill_value=0
)

# Get actual years with data (no empty years)
years = sorted(pivot.columns.tolist())

# Sort by total count (no minimum threshold)
pivot['total'] = pivot.sum(axis=1)
pivot = pivot.sort_values('total', ascending=False)

country_totals = pivot['total'].copy()
pivot = pivot.drop('total', axis=1)

# Update country names: add (China) for Hong Kong and Taiwan, and (the Mainland) for China
country_names = []
for country in pivot.index:
    if country == 'Hong Kong':
        country_names.append('Hong Kong (China)')
    elif country == 'Taiwan':
        country_names.append('Taiwan (China)')
    elif country == 'China':
        country_names.append('China (the Mainland)')
    else:
        country_names.append(country)


# Create figure
num_years = len(years)
num_countries = len(pivot)

circle_diameter = 0.05  # inches
spacing = 1.0  # data coordinate spacing

fig_width = num_years * circle_diameter + 1.8
fig_height = num_countries * circle_diameter + 1.0

fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=500)
ax.set_facecolor('white')

# Color mapping
cmap = plt.cm.YlOrRd
max_count = pivot.max().max()
norm = plt.Normalize(vmin=0, vmax=max_count)

circle_radius = 0.48

# Draw circles
for i, country in enumerate(pivot.index):
    for j, year in enumerate(years):
        count = pivot.loc[country, year]
        if count > 0:
            color = cmap(norm(count))
            circle = plt.Circle((j * spacing, (num_countries - 1 - i) * spacing), 
                              radius=circle_radius,
                              color=color,
                              alpha=0.92,
                              edgecolor='white',
                              linewidth=0.1)
            ax.add_patch(circle)

# Set axis limits
ax.set_xlim(-0.8 * spacing, (num_years - 0.2) * spacing)
ax.set_ylim(-0.6 * spacing, (num_countries - 0.4) * spacing)
ax.set_aspect('equal', adjustable='box')

# X-axis: years
ax.set_xticks([i * spacing for i in range(num_years)])
ax.set_xticklabels([int(y) for y in years], rotation=90, ha='center', fontsize=4)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.tick_params(axis='x', pad=0.3, length=1, width=0.25)

# Y-axis: country names (with China notation)
ax.set_yticks([i * spacing for i in range(num_countries)])
ax.set_yticklabels(country_names[::-1], fontsize=4)
ax.tick_params(axis='y', pad=0.3, length=1, width=0.25)

# Add total counts next to country names
for i, country in enumerate(pivot.index[::-1]):
    total = country_totals[country]
    ax.text((num_years + 0.15) * spacing, i * spacing, f'({int(total)})', 
           fontsize=3.5, va='center', color='gray')

# Add vertical lines every 5 years
for i in range(0, num_years, 5):
    ax.axvline(x=(i - 0.5) * spacing, color='lightgray', 
              linewidth=0.2, alpha=0.25, linestyle='--')

# Show top and left spines
ax.spines['top'].set_visible(True)
ax.spines['top'].set_color('black')
ax.spines['top'].set_linewidth(0.5)

ax.spines['left'].set_visible(True)
ax.spines['left'].set_color('black')
ax.spines['left'].set_linewidth(0.5)

ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm._A = []
cbar_ax = fig.add_axes([0.15, 0.015, 0.7, 0.008])
cbar = plt.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Number of New Data Centers', fontsize=5, labelpad=1.5)
cbar.ax.tick_params(labelsize=4, width=0.25, length=1.2, pad=0.8)
cbar.outline.set_linewidth(0.25)

plt.subplots_adjust(left=0.11, right=0.96, top=0.97, bottom=0.04)

# Save PDF only
output_path = os.path.join(FIGURES, 'si_fig1.pdf')
plt.savefig(output_path, bbox_inches='tight')

plt.show()

print(f"\nFigure saved to: {output_path}")
